# core
> Manage `.gheasy/config.json`, generate GitHub Actions workflows, and run `gh_*` CI/CD helpers.

`gheasy` — Git + GitHub Easy. A declarative config package where `.gheasy/config.json` is the
source of truth for CI/CD workflows, git hooks, LFS patterns, branch protection, and repo settings.
Analogous to how `Dockerfile` configures containers or `Caddyfile` configures a reverse proxy:
edit the config once, regenerate all derived files from it.

Built on `ghapi` (GitHub REST API) from the AnswerDotAI ecosystem. `gheasy` provides
the higher-level workflow layer that sits between raw API calls and use-case-specific tools.


In [ ]:
#| default_exp core

## Setup & Imports

In [ ]:
#| export
from __future__ import annotations
import re, subprocess as sp, stat, os, yaml
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from typing import Any
from ghapi.all import GhApi
from fastcore.all import store_attr, patch, filter_values, true, Path, filter_keys, in_, L, listify, not_, is_
from gheasy.workflow import Workflow, JobBuilder, StepBuilder

### Test Helper

In [ ]:
import tempfile, shutil
class GheasyTestRepo:
    'Context manager: real temp git repo with initialized git and fake remote.'
    def __enter__(self):
        self._tmp = tempfile.mkdtemp()
        self.path = Path(self._tmp)
        # bare repo to push to
        self._bare = tempfile.mkdtemp()
        sp.run(['git', 'init', '--bare', self._bare], check=True, capture_output=True)
        self.remote = f'file://{self._bare}'
        sp.run(['git', 'init', str(self.path)], check=True, capture_output=True)
        sp.run(['git', '-C', str(self.path), 'remote', 'add', 'origin', self.remote], check=True, capture_output=True)
        sp.run(['git', '-C', str(self.path), 'config', 'user.email', 'test@test.com'], check=True, capture_output=True)
        sp.run(['git', '-C', str(self.path), 'config', 'user.name', 'Test'], check=True, capture_output=True)
        return self
    def __exit__(self, *a):
        shutil.rmtree(self._tmp, ignore_errors=True)
        shutil.rmtree(self._bare, ignore_errors=True)

In [ ]:
with GheasyTestRepo() as t:
    assert t.path.exists()
    assert (t.path / '.git').is_dir()
    assert t.remote.startswith('file://')
    print('GheasyTestRepo OK')

GheasyTestRepo OK


## Config

In [ ]:
#| export
@dataclass
class EnvConfig:
    host: str
    domain: str
    branch: str = 'main'
    srv_path: str = '/srv/app'

@dataclass
class DeployOptions:
    user: str = 'deploy'
    key: str = None
    build: bool = True
    exclude: list = field(default_factory=list)
    extra_dirs: list = field(default_factory=list)
    health_timeout: int = 60

@dataclass
class GheasyConfig:
    """Mutable config backed by `.gheasy/config.json`.
    Chainable mutating methods return `self` for fluent use.
    Use `GheasyConfig.load(path)` to read; `.save(path)` to write.
    Not thread-safe: do not share instances across concurrent callers."""
    app: str = ''
    envs: dict = field(default_factory=dict)           # {env_name: EnvConfig}
    health_path: str = '/health'
    test_cmd: str = None
    env_schema: dict = field(default_factory=dict)     # {KEY: None} → secret, {KEY: "val"} → variable
    hooks: dict = field(default_factory=dict)
    lfs: list = field(default_factory=list)
    workflows: dict = field(default_factory=dict)      # opt-in flags, all False by default
    workflow_preset: str = None                        # 'python'|'fasthtml'|'fastapi-react'|'nodejs'|'rust'|'go'
    deploy_cmd: str = None                             # override deploy step; None = skip deploy job
    extra_jobs: dict = field(default_factory=dict)
    deploys: dict = field(default_factory=dict)        # runtime state, persisted but not in workflow YAML
    deploy_pkg: str = None

@patch
def add_env(self:GheasyConfig, name, host, domain, branch=None, srv_path='/srv/app'):
    self.envs[name] = EnvConfig(host=host, domain=domain, branch=branch or 'main', srv_path=srv_path)
    return self

@patch
def set_lfs(self:GheasyConfig, patterns):
    self.lfs = list(patterns)
    return self

@patch
def set_hooks(self:GheasyConfig, hooks):
    self.hooks = hooks
    return self

@patch
def set_workflows(self:GheasyConfig, workflows):
    self.workflows = {**self.workflows, **workflows}
    return self

@patch
def add_job(self:GheasyConfig, name, job):
    self.extra_jobs[name] = job
    return self

@patch
def to_dict(self:GheasyConfig): return asdict(self)

@patch(cls_method=True)
def from_dict(cls:GheasyConfig, d):
    envs = {k: EnvConfig(**v) for k, v in d.get('envs', {}).items()}
    rest = filter_keys(d, lambda k: k in cls.__dataclass_fields__ and k != 'envs')
    return cls(envs=envs, **rest)

@patch
def save(self:GheasyConfig, path='.'):
    cfg_path(path).write_json(self.to_dict())
    return self

@patch(cls_method=True)
def load(cls:GheasyConfig, path='.'):
    'Load config from .gheasy/config.json. Raises FileNotFoundError if absent.'
    return cls.from_dict(cfg_path(path).read_json()) if cfg_path(path).exists() else cls()

In [ ]:
#| export
@patch(cls_method=True)
def from_pyproject(cls:GheasyConfig, path='.'):
    "Bootstrap config from pyproject.toml [project] name."
    import tomllib
    with open(Path(path)/'pyproject.toml', 'rb') as f: data = tomllib.load(f)
    return cls(app=data['project']['name'])

In [ ]:
with GheasyTestRepo() as t:
    (t.path / 'pyproject.toml').write_text('[project]\nname = "my-cool-app"\n')
    cfg = GheasyConfig.from_pyproject(path=t.path)
    assert cfg.app == 'my-cool-app'
    print('from_pyproject OK')

from_pyproject OK


In [ ]:
#| export
def cfg_path(path='.') -> Path:
    'Return path to .gheasy/config.json'
    return Path(path) / '.gheasy' / 'config.json'

In [ ]:
with GheasyTestRepo() as t:
    cfg = GheasyConfig(app='myapp', workflow_preset='fasthtml', deploy_cmd='./deploy.sh')
    cfg.add_env('prod', '1.2.3.4', 'myapp.com').save(t.path)
    assert cfg_path(t.path).exists()
    loaded = GheasyConfig.load(t.path)
    assert loaded.app == 'myapp'
    assert loaded.envs['prod'].host == '1.2.3.4'
    assert loaded.envs['prod'].domain == 'myapp.com'
    assert loaded.workflow_preset == 'fasthtml'
    assert loaded.deploy_cmd == './deploy.sh'
    print('round-trip with new fields OK')

round-trip with new fields OK


In [ ]:
with GheasyTestRepo() as t:
    cfg = (GheasyConfig(app='test').add_env('prod', '1.2.3.4', 'test.com')
           .set_lfs(['*.pt', '*.onnx']).set_hooks({'pre-commit': 'uv run nbdev_prepare'})
           .save(t.path))
    loaded = GheasyConfig.load(t.path)
    assert loaded.lfs == ['*.pt', '*.onnx']
    assert 'pre-commit' in loaded.hooks
    print('chainable mutations OK')

chainable mutations OK


In [ ]:
cfg = GheasyConfig.from_dict({'app': 'test', 'unknown_key': 99, 'envs': {}})
assert cfg.app == 'test'
assert not hasattr(cfg, 'unknown_key')
print('from_dict unknown key filter OK')

from_dict unknown key filter OK


## Generators

In [ ]:
#| export
_preset = {'python': {}, 'fasthtml': {}, 'rust': {'rust': True}, 'go': {'go': True},
'fastapi-react': {'node': True}, 'nodejs': {'node': True}}

def _res_wfs(cfg):
    "Merge preset defaults with explicit cfg.workflows (explicit always wins)."
    base = _preset.get(cfg.workflow_preset, {}) if cfg.workflow_preset else {}
    return {**base, **cfg.workflows}

def mk_deploy_job(env_name, cfg, wfb, cmd=None):
    "Add deploy job for env_name to wfb using DSL. Returns None (with warning) if no deploy cmd."
    env = cfg.envs[env_name]
    deploy_step = cmd or cfg.deploy_cmd
    if not deploy_step:
        print(f'Warning: no deploy_cmd configured for {env_name!r} — skipping deploy job. '
              f'Set cfg.deploy_cmd or pass cmd= to mk_deploy_job.')
        return None
    (wfb.job(f'deploy-{env_name}').runs_on('ubuntu-latest').needs('test').environment(env_name)
        .if_(f"github.ref == 'refs/heads/{env.branch}'").checkout().end_step()
        .setup_uv().end_step()
        .step('Deploy SSH key').run(
            'mkdir -p ~/.ssh && echo "${{ secrets.DEPLOY_KEY }}" > ~/.ssh/id_rsa '
            '&& chmod 600 ~/.ssh/id_rsa\n'
            f'ssh-keyscan {env.host} >> ~/.ssh/known_hosts').end_step()
        .step(f'Deploy to {env_name}').run(deploy_step).end_job())

In [ ]:
from fastcore.test import test_eq

# No deploy cmd → None returned, job not added to wfb
cfg_no_cmd = GheasyConfig(app='myapp')
cfg_no_cmd.add_env('prod', '1.2.3.4', 'myapp.com')
wfb_no = Workflow('test')
wfb_no.on.push()
result = mk_deploy_job('prod', cfg_no_cmd, wfb_no)
assert result is None
assert 'deploy-prod' not in wfb_no._job_map

# cmd= takes priority over cfg.deploy_cmd
cfg_cmd = GheasyConfig(app='myapp', deploy_cmd='cfg_deploy.sh')
cfg_cmd.add_env('prod', '1.2.3.4', 'myapp.com')
wfb_cmd = Workflow('test')
mk_deploy_job('prod', cfg_cmd, wfb_cmd, cmd='override.sh')
jb = wfb_cmd._job_map['deploy-prod']
deploy_step = next(s for s in jb._steps if s._data.get('name') == 'Deploy to prod')
test_eq(deploy_step._data['run'], 'override.sh')

# cfg.deploy_cmd used when cmd not passed
wfb2 = Workflow('test')
mk_deploy_job('prod', cfg_cmd, wfb2)
jb2 = wfb2._job_map['deploy-prod']
deploy_step2 = next(s for s in jb2._steps if s._data.get('name') == 'Deploy to prod')
test_eq(deploy_step2._data['run'], 'cfg_deploy.sh')

# SSH key step always present
ssh = next(s for s in jb2._steps if s._data.get('name') == 'Deploy SSH key')
assert 'id_rsa' in ssh._data['run']

# Job has correct runner, needs, environment, branch gate
test_eq(jb2._data['runs-on'], 'ubuntu-latest')
test_eq(jb2._data['needs'], 'test')
test_eq(jb2._data['environment'], 'prod')
test_eq(jb2._data['if'], "github.ref == 'refs/heads/main'")

# uv shortcuts used: setup-uv step present (no manual uses/step call)
setup_uv_step = next(s for s in jb2._steps if s._data.get('uses') == 'astral-sh/setup-uv@v5')
assert setup_uv_step is not None

print('mk_deploy_job OK')

mk_deploy_job OK


## Workflow generation

`mk_workflow()` generates a complete GitHub Actions YAML from config. It supports four built-in job
presets controlled by `cfg.workflows` flags:

| Flag | Job added | What it does |
|---|---|---|
| `lint: true` | `lint` | `ruff check` + `ruff format --check` |
| `publish_pypi: true` | `publish` | Build + OIDC upload to PyPI on releases |
| `docker_build: true` | `docker` | Build + push to GHCR |
| `on_pull_request: true` | — | Adds `pull_request` trigger alongside `push` |

Custom jobs go in `cfg.extra_jobs` (added via `add_job()` or `gh_add_job()`). The generated YAML
is written to `.github/workflows/gheasy.yml` — never hand-edit it, regenerate with `gh_workflow()`.

In [ ]:
#| export
def _inject_raw_job(job_id, wfb, raw):
    "Convert a raw job dict into a JobBuilder attached to wfb."
    jb, steps = JobBuilder(job_id, wfb), raw.pop('steps', [])
    jb._data.update(raw)
    for sd in steps:
        sb = StepBuilder(sd.get('name', ''), jb)
        sb._data = sd
        jb._steps.append(sb)
    wfb._jobs.append(jb)
    wfb._job_map[job_id] = jb

In [ ]:
#| export
def mk_workflow(cfg, *,
    on_pull_request: bool | None = None,    # also trigger on pull_request events
    test:            bool | None = None,    # uv test job (pytest by default, cfg.test_cmd to override)
    lint:            bool | None = None,    # ruff check + format (needs: test)
    publish_pypi:    bool | None = None,    # PyPI trusted publishing on release (needs: test)
    docker_build:    bool | None = None,    # Docker build + push to GHCR (needs: test)
    node:            bool | None = None,    # Node.js build + test (frontend job)
    rust:            bool | None = None,    # Rust build + test + clippy
    go:              bool | None = None,    # Go build + test + vet
):
    '''Generate complete GitHub Actions workflow YAML from a GheasyConfig.
    Each flag enables a corresponding job. `None` (default) defers to the
    value resolved from `cfg.workflow_preset` + `cfg.workflows`. Passing
    `True` or `False` overrides cfg for that flag.
    Example::
        cfg = GheasyConfig.from_pyproject().add_env('prod', '1.2.3.4', 'myapp.com')
        cfg.deploy_cmd = './deploy.sh'
        print(mk_workflow(cfg, test=True, lint=True))
    '''
    kw = dict(on_pull_request=on_pull_request, test=test, lint=lint, publish_pypi=publish_pypi,
             docker_build=docker_build, node=node, rust=rust, go=go)
    flags = _res_wfs(cfg) | filter_values(kw, not_(is_(None)))
    branches = list({e.branch for e in cfg.envs.values()}) or ['main']
    wfb = Workflow(f'{cfg.app} CI')
    wfb.on.push(branches=branches)
    if flags.get('on_pull_request'): wfb.on.pull_request(branches=branches)
    if flags.get('test'): wfb.uv_test_job(**({'test_cmd': cfg.test_cmd} if cfg.test_cmd else {}))
    if flags.get('lint'): wfb.uv_lint_job()
    if flags.get('publish_pypi'): wfb.uv_pypi_job()
    if flags.get('docker_build'): wfb.docker_job(cfg.app)
    if flags.get('node'): wfb.node_job()
    if flags.get('rust'): wfb.rust_job()
    if flags.get('go'): wfb.go_job()
    for env_name in cfg.envs: mk_deploy_job(env_name, cfg, wfb)
    for name, job in cfg.extra_jobs.items(): _inject_raw_job(name, wfb, dict(job))
    return '# Managed by gheasy — do not edit directly\n' + wfb.build().to_yaml()

In [ ]:
# deploy skipped when no deploy_cmd
cfg_no_deploy = GheasyConfig(app='myapp')
cfg_no_deploy.add_env('prod', '1.2.3.4', 'myapp.com')
wf_str = mk_workflow(cfg_no_deploy)
assert 'deploy-prod' not in wf_str
assert '# Managed by gheasy' in wf_str

# deploy included when deploy_cmd set
cfg = GheasyConfig(app='myapp', deploy_cmd='./deploy.sh')
cfg.add_env('prod', '1.2.3.4', 'myapp.com')
wf_str2 = mk_workflow(cfg)
assert 'deploy-prod' in wf_str2

# explicit flag=True overrides cfg (no flags in cfg, test=True via kwarg)
wf_test = mk_workflow(cfg, test=True)
assert 'uv run pytest' in wf_test
assert 'jobs:\n  test:' in wf_test or 'test:' in wf_test

# explicit flag=False suppresses even if cfg would enable it
cfg_with_test = GheasyConfig(app='myapp', workflows={'test': True})
cfg_with_test.add_env('prod', '1.2.3.4', 'myapp.com')
cfg_with_test.deploy_cmd = './deploy.sh'
wf_notests = mk_workflow(cfg_with_test, test=False)
assert 'uv run pytest' not in wf_notests

# lint + test via kwargs
wf_lint = mk_workflow(cfg, test=True, lint=True)
assert 'ruff' in wf_lint
assert 'needs: test' in wf_lint or 'needs: test' in wf_lint.replace("'", "")

# on_pull_request adds pull_request trigger
wf_pr = mk_workflow(cfg, on_pull_request=True)
assert 'pull_request:' in wf_pr

# workflow_preset='rust' enables rust job by default
cfg_rust = GheasyConfig(app='myrust', workflow_preset='rust')
wf_rust = mk_workflow(cfg_rust)
assert 'dtolnay/rust-toolchain' in wf_rust

print('mk_workflow OK')

mk_workflow OK


## .gitattributes generation

`mk_gitattributes()` generates the managed LFS block for `.gitattributes`. It wraps content in
`# BEGIN gheasy-lfs` / `# END gheasy-lfs` sentinels so `gh_gitattributes()` can safely update
only the managed section without touching user-added lines.

`mk_dependabot()` generates `.github/dependabot.yml` from config for automated dependency updates.

In [ ]:
#| export
_lfs_st, _lfs_end = '# BEGIN gheasy-lfs', '# END gheasy-lfs'

def mk_gitattributes(patterns):
    'Generate LFS block for .gitattributes with sentinel comments.'
    st, end = _lfs_st, _lfs_end
    ga = L(patterns).map(lambda p: f'{p} filter=lfs diff=lfs merge=lfs -text')
    return '\n'.join(st+ga+end) + '\n'

def mk_dependabot(ecosystems=('pip', 'github-actions'), schedule='weekly', assignees=None, lbls=None):
    'Generate .github/dependabot.yml content.'
    opt = dict(assignees=listify(assignees)) if assignees else {} | dict(labels=listify(lbls)) if lbls else {}
    def upd(eco): return {'package-ecosystem': eco, 'directory': '/', 'schedule': {'interval': schedule}}|opt
    return yaml.dump(dict(version=2,updates=L(ecosystems).map(upd)), sort_keys=False)

def mk_hook(cmd):
    'Generate a bash hook script with gheasy-managed sentinels.'
    return f'''
    #!/bin/bash
    # BEGIN gheasy-managed
    {cmd}
    # END gheasy-managed
    '''.strip() + '\n'

In [ ]:
# mk_gitattributes
out = mk_gitattributes(['*.pt', '*.onnx'])
assert '# BEGIN gheasy-lfs' in out
assert '# END gheasy-lfs' in out
assert '*.pt filter=lfs diff=lfs merge=lfs -text' in out

# mk_hook
h = mk_hook('uv run nbdev_prepare')
assert '#!/bin/bash' in h
assert 'gheasy-managed' in h
assert 'uv run nbdev_prepare' in h

# mk_workflow — no lint by default
cfg = GheasyConfig(app='myapp', workflow_preset='python')
cfg.add_env('prod', '1.2.3.4', 'myapp.com')
wf = mk_workflow(cfg)
assert '# Managed by gheasy' in wf
assert 'myapp CI' in wf
assert 'lint' not in wf

# mk_workflow — lint opt-in
cfg2 = GheasyConfig(app='myapp', workflow_preset='python', workflows={'lint': True})
cfg2.add_env('prod', '1.2.3.4', 'myapp.com')
assert 'lint' in mk_workflow(cfg2)

# preset override — disable test
cfg3 = GheasyConfig(app='myapp', workflow_preset='fasthtml', workflows={'test': False})
cfg3.add_env('prod', '1.2.3.4', 'myapp.com')
assert 'test:' not in mk_workflow(cfg3)
print('generators OK')

generators OK


## Git Ops

## Git hooks

`gh_hooks()` installs local git hooks from a dict `{hook_name: script_body}`. It writes to
`.git/hooks/` and uses `# BEGIN gheasy-managed` / `# END gheasy-managed` sentinels so:
- Re-running updates only the managed block without touching other hook content
- The user can add their own hook logic above/below the managed block

`gh_githooks_pre_commit()` is a convenience wrapper for the most common nbdev hook.

**Why not use pre-commit?** For projects already using `uv`/`nbdev`, a direct shell hook is
simpler and doesn't require installing the `pre-commit` Python package. Use `pre-commit` if
you need cross-language hooks or hook sharing across many repos.

**Tracked hooks:** Hooks live in `.git/hooks/` which git doesn't track. To commit hooks to the
repo, run `git config core.hooksPath .githooks` and point `gh_hooks()` at `.githooks/` instead:
```python
gh_hooks({'pre-commit': 'uv run nbdev_prepare'}, path='.')
# Then: git add .githooks/ && git commit -m 'chore: add git hooks'
```

In [ ]:
#| export
def nbdev_hook():
    "Return the pre-commit cmd that runs nbdev_prepare via uv, injecting nbdev if not in project deps."
    return 'uv run --with nbdev nbdev-prepare'

def gh_hooks(hooks=None, path='.'):
    """Install git hooks. hooks = {'pre-commit': cmd, ...} where cmd is a str or list[str].
    Reads cfg.hooks if hooks not passed."""
    hooks = hooks or GheasyConfig.load(path).hooks
    if not hooks: return
    if not (git_dir := Path(path) / '.git').exists(): print(f'no .git directory at {path!r}.'); return
    hooks_dir = git_dir / 'hooks'
    hooks_dir.mkdir(exist_ok=True)
    for name, cmd in hooks.items():
        if isinstance(cmd, list): cmd = '\n'.join(cmd)
        hp = hooks_dir / name
        ex, block = hp.read_text() if hp.exists() else '', mk_hook(cmd)
        sentinel_re = re.compile(r'# BEGIN gheasy-managed.*?# END gheasy-managed\n', re.DOTALL)
        if sentinel_re.search(ex): nc = sentinel_re.sub(block, ex)
        else: nc = ex + ('\n' if ex and not ex.endswith('\n') else '') + block
        hp.write_text(nc)
        hp.chmod(hp.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

def gh_githooks_pre_commit(cmd=None, path='.'):
    "Install pre-commit hook. Defaults to nbdev_hook(). Pass cmd= to override."
    gh_hooks({'pre-commit': cmd or nbdev_hook()}, path=path)

In [ ]:
import tempfile, shutil

In [ ]:

# No .git dir → warns and returns (no error)
tmp = Path(tempfile.mkdtemp())
try: gh_hooks({'pre-commit': 'echo hi'}, path=tmp)  # should print warning, not raise
finally: shutil.rmtree(tmp)

# Missing hooks/ subdir → created automatically (rmtree to clear git init's sample files)
with GheasyTestRepo() as t:
    shutil.rmtree(t.path / '.git' / 'hooks')
    gh_hooks({'pre-commit': 'echo hi'}, path=t.path)
    assert (t.path / '.git' / 'hooks' / 'pre-commit').exists()

# No arg → reads from cfg.hooks
with GheasyTestRepo() as t:
    GheasyConfig(app='myapp', hooks={'pre-commit': 'echo test'}).save(t.path)
    gh_hooks(path=t.path)
    content = (t.path / '.git' / 'hooks' / 'pre-commit').read_text()
    assert 'echo test' in content

# No arg, no config → warns and returns (no error)
with GheasyTestRepo() as t: gh_hooks(path=t.path)

# list[str] cmd → joined with newlines, all commands present in hook
with GheasyTestRepo() as t:
    gh_hooks({'pre-commit': ['uv run ruff check .', 'uv run nbdev-prepare']}, path=t.path)
    content = (t.path / '.git' / 'hooks' / 'pre-commit').read_text()
    assert 'uv run ruff check .' in content
    assert 'uv run nbdev-prepare' in content

# nbdev_hook() returns the --with nbdev form (hyphenated nbdev-prepare)
assert 'nbdev-prepare' in nbdev_hook()
assert '--with nbdev' in nbdev_hook()

# gh_githooks_pre_commit defaults to nbdev_hook()
with GheasyTestRepo() as t:
    gh_githooks_pre_commit(path=t.path)
    content = (t.path / '.git' / 'hooks' / 'pre-commit').read_text()
    assert '--with nbdev' in content

print('gh_hooks OK')

no .git directory at Path('/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmptcl1ltg7').
gh_hooks OK


## LFS and .gitattributes

`gh_lfs()` runs `git lfs install` + `git lfs track` for each pattern and persists patterns to
config so `.gitattributes` can be regenerated at any time without re-running `git lfs track`.

`gh_gitattributes()` regenerates `.gitattributes` from `cfg.lfs` — **merge-safe**: it only
updates the `# BEGIN gheasy-lfs` / `# END gheasy-lfs` block, leaving all other lines untouched.
This means user-added attributes like `*.sh text eol=lf` are always preserved.

`gh_push_env()` routes env vars to GitHub Secrets or Variables based on `env_schema`: `null`
values become secrets, string-default values become variables.

**Typical LFS workflow:**
```python
gh_lfs(['*.onnx', '*.db', '*.safetensors'])  # run once per repo
# Then after any config change:
gh_gitattributes()  # regenerate .gitattributes from config
# git add .gitattributes && git commit -m 'chore: update .gitattributes'
```

In [ ]:
#| export
_LFS_DEFAULTS = ['*.db', '*.onnx', '*.bin', '*.pt', '*.pkl', '*.safetensors', '*.tar.gz', '*.mp4', '*.parquet']

def gh_lfs(patterns=None, persist=True, path='.'):
    "Track LFS patterns. If persist=True, saves to config (source of truth for gh_gitattributes)."
    patterns = patterns or _LFS_DEFAULTS
    sp.run(['git', 'lfs', 'install'], cwd=path, check=True, capture_output=True)
    for p in patterns: sp.run(['git', 'lfs', 'track', p], cwd=path, check=True, capture_output=True)
    if persist: GheasyConfig.load(path).set_lfs(patterns).save(path)

def gh_gitattributes(path='.'):
    """Regenerate gheasy-managed LFS block in .gitattributes from config.
    Preserves all non-managed lines. Call after gh_lfs(persist=True)."""
    if not cfg_path(path).exists(): print('No .gheasy/config.json — run gh_init first.'); return
    cfg = GheasyConfig.load(path)
    if not cfg.lfs: print('No LFS patterns in config — run gh_lfs(persist=True) first.'); return
    ga_path = Path(path) / '.gitattributes'
    existing = ga_path.read_text() if ga_path.exists() else ''
    block = mk_gitattributes(cfg.lfs)
    sentinel_re = re.compile(r'# BEGIN gheasy-lfs.*?# END gheasy-lfs\n', re.DOTALL)
    new_content = sentinel_re.sub(block, existing) if sentinel_re.search(existing) else existing + block
    ga_path.write_text(new_content)

In [ ]:
# gh_hooks installs hook and makes it executable
with GheasyTestRepo() as t:
    gh_hooks({'pre-commit': 'uv run nbdev_prepare'}, path=t.path)
    hook_path = t.path / '.git' / 'hooks' / 'pre-commit'
    assert hook_path.exists()
    text = hook_path.read_text()
    assert '# BEGIN gheasy-managed' in text
    assert 'uv run nbdev_prepare' in text
    assert hook_path.stat().st_mode & stat.S_IXUSR

# re-running updates only managed block, preserves user content
with GheasyTestRepo() as t:
    (t.path / '.git' / 'hooks' / 'pre-commit').write_text('# user content\n')
    gh_hooks({'pre-commit': 'uv run nbdev_prepare'}, path=t.path)
    text = (t.path / '.git' / 'hooks' / 'pre-commit').read_text()
    assert '# user content' in text
    assert 'uv run nbdev_prepare' in text
    gh_hooks({'pre-commit': 'uv run pytest'}, path=t.path)
    text2 = (t.path / '.git' / 'hooks' / 'pre-commit').read_text()
    assert 'uv run pytest' in text2
    assert text2.count('BEGIN gheasy-managed') == 1

# gh_gitattributes writes and updates managed block
with GheasyTestRepo() as t:
    GheasyConfig(app='test').set_lfs(['*.pt', '*.onnx']).save(t.path)
    gh_gitattributes(path=t.path)
    ga = (t.path / '.gitattributes').read_text()
    assert '# BEGIN gheasy-lfs' in ga
    assert '*.pt filter=lfs' in ga

print('git ops OK')

git ops OK


## GitHub Ops

## Private helpers

These internal helpers are used across `gh_*` functions.

- `_git_branch()`: detect current branch via `git rev-parse`
- `_get_repo_slug()`: parse `(owner, repo)` from git remote origin URL
- `_resolve_gh_token()`: tiered token resolution — explicit → `GITHUB_TOKEN` env → `.fastops/token` file → `FASTSHIP_TOKEN` env
- `_gh_api()`: returns `(owner, repo_name, GhApi)` for the repo at path
- `gh_record_deploy()`: write deploy state back to config (called by CI after successful deploy)

## Repo configuration

`gh_protect()` sets branch protection rules via ghapi — require reviews, status checks, dismiss
stale reviews, restrict pushes, etc.

`gh_topics()` replaces all repo topics via ghapi — useful for keeping topic metadata in sync
with the config.

Both require a GitHub token with `repo` admin scope. Token resolution: explicit string →
`GITHUB_TOKEN` env → `.fastops/token` file → `FASTSHIP_TOKEN` env.

**Branch protection example:**
```python
gh_protect('main',
    require_reviews=1,
    require_status_checks=['test', 'lint'],
    dismiss_stale=True)
```

In [ ]:
#| export
def _resolve_gh_token(token=None):
    "Tiered resolution: explicit > GITHUB_TOKEN env"
    return token or os.getenv('GITHUB_TOKEN')

def _get_repo_slug(path='.'):
    "Parse (owner, repo) from git remote URL."
    r = sp.run(['git', 'remote', 'get-url', 'origin'], cwd=path, capture_output=True, text=True)
    url = r.stdout.strip()
    m = re.search(r'[:/]([^/]+)/([^/]+?)(?:\.git)?$', url)
    if not m: raise ValueError(f'Cannot parse owner/repo from remote: {url}')
    return m.group(1), m.group(2)

def _gh_api(token=None, path='.'):
    "Return (owner, repo_name, GhApi) instance."
    owner, repo = _get_repo_slug(path)
    return owner, repo, GhApi(owner=owner, repo=repo, token=token)

def gh_protect(branch='main', require_reviews=1, dismiss_stale=True, require_status_checks=('test',),
           enforce_admins=False, allow_force_pushes=False, token=None, path='.'):
    "Set branch protection rules via ghapi. Requires token with repo admin scope."
    owner, repo, api = _gh_api(token or _resolve_gh_token(), path)
    rsc = dict(strict=True, contexts=listify(require_status_checks))
    rprr=dict(required_approving_review_count=require_reviews, dismiss_stale_reviews=dismiss_stale)
    api.repos.update_branch_protection(branch=branch, enforce_admins=enforce_admins, required_status_checks=rsc,
        required_pull_request_reviews=rprr, restrictions=None, allow_force_pushes=allow_force_pushes)

def gh_topics(topics, token=None, path='.'):
    "Replace repo topics. GitHub enforces lowercase."
    owner, repo, api = _gh_api(_resolve_gh_token(token), path)
    api.repos.replace_all_topics(names=[t.lower() for t in topics])

## Secrets management

`gh_secret()` sets a single GitHub Actions secret via the `gh` CLI. Supports scoping to a
GitHub Environment (e.g. `env='prod'`) for environment-specific secrets.

`gh_secrets_from_file()` reads a `.env` file and pushes every key as a secret — simpler than
`gh_push_env()` when you don't need mixed secret/variable routing.

**Comparison:**
- `gh_secret()` / `gh_secrets_from_file()` — every key is a secret, no schema needed
- `gh_push_env()` — routes keys to secrets vs variables based on `env_schema`

**Setup:** Requires `gh` CLI authenticated (`gh auth login`). No Python deps needed.

In [ ]:
#| export
def gh_secret(key, value, env=None, dry_run=False):
    "Set a single GitHub secret, optionally scoped to an environment."
    cmd = ['gh', 'secret', 'set', key, '--body', value]
    if env: cmd += ['--env', env]
    if dry_run: print(f'[dry-run] {" ".join(cmd)}'); return
    sp.run(cmd, check=True)

def gh_push_env(envs, dry_run=False, path='.'):
    """Route env vars to secrets or variables using cfg.env_schema.
    Schema: {KEY: None} -> Secret, {KEY: 'default'} -> Variable."""
    if not cfg_path(path).exists(): raise FileNotFoundError('No .gheasy/config.json')
    cfg = GheasyConfig.load(path)
    schema = cfg.env_schema
    for k in filter_keys(envs, not_(in_(schema))): print(f'Warning: {k} not in env_schema, skipping')
    for k,v in filter_keys(envs, in_(schema)).items():
         if schema[k] is None: gh_secret(k, v, dry_run=dry_run)
         else:
             cmd = ['gh', 'variable', 'set', k, '--body', v]
             if dry_run: print(f'[dry-run] {" ".join(cmd)}')
             else: sp.run(cmd, check=True)

def gh_secrets_from_file(env_file='.env', env=None, dry_run=False):
    "Read KEY=VALUE pairs from env_file and push all as GitHub secrets."
    lines = Path(env_file).read_text().splitlines()
    for line in lines:
        line = line.strip()
        if not line or line.startswith('#'): continue
        key, _, value = line.partition('=')
        gh_secret(key.strip(), value.strip(), env=env, dry_run=dry_run)

In [ ]:
#| export
def gh_deploy_key_setup(key_path, dry_run=False):
    """Read SSH private key from file and push as DEPLOY_KEY GitHub secret.

    Step 3 of deploy key setup (steps 1-2 require manual SSH access):
    1. Generate key pair: ssh-keygen -t ed25519 -f deploy_key
    2. Add public key to server ~/.ssh/authorized_keys (manual)
    3. Call this function with the private key path to upload to GitHub secrets
    """
    key = Path(key_path).read_text()
    gh_secret('DEPLOY_KEY', key, dry_run=dry_run)
    if not dry_run: print(f'DEPLOY_KEY set from {key_path}')

In [ ]:
import io, contextlib

# gh_push_env dry-run routes correctly
with GheasyTestRepo() as t:
    cfg = GheasyConfig(app='test', env_schema={'DB_URL': None, 'APP_ENV': 'dev'})
    cfg.save(t.path)
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        gh_push_env({'DB_URL': 'postgres://...', 'APP_ENV': 'prod'}, dry_run=True, path=t.path)
    out = buf.getvalue()
    assert 'secret set DB_URL' in out
    assert 'variable set APP_ENV' in out

print('github ops OK')

github ops OK


## Lifecycle

## _resolve_gh_repo_input

Normalise any of the three repo address forms to `(owner, repo)`:

- `owner/repo` string
- GitHub URL (`https://github.com/owner/repo`)
- Local path with a `origin` remote pointing to GitHub

In [ ]:
#| export
def _git_branch(path='.'):
    "Detect current git branch, defaulting to 'main'."
    r = sp.run(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], cwd=path, capture_output=True, text=True)
    return r.stdout.strip() or 'main'

def _resolve_gh_repo_input(ref=None, path='.'):
    "Normalize to (owner, repo). Accepts 'owner/repo', GitHub URL, or local path."
    if ref is None: return _get_repo_slug(path)
    if '/' in ref and 'github.com' not in ref: return tuple(ref.split('/', 1))
    m = re.search(r'github\.com[:/]([^/]+)/([^/]+?)(?:\.git)?$', ref)
    if m: return m.group(1), m.group(2)
    raise ValueError(f'Cannot parse owner/repo from: {ref}')

## gh_* — manage `.fastops/config.json`

These are the primary CLI-style entry points. All read/write `.fastops/config.json` and
print actionable instructions.

| Function | What it does |
|---|---|
| `gh_init()` | Create/update config. Auto-detects branch and pytest. |
| `gh_add_env()` | Add/update an environment + regenerate workflow locally. |
| `gh_workflow()` | Push/write `.github/workflows/gheasy.yml`. |
| `gh_status()` | Print deploy state for all environments. |
| `gh_add_job()` | Add a custom CI job to config. |

In [ ]:
#| export
def gh_init(name, host, domain, env='prod', branch=None, srv_path='/srv/app',
            health_path='/health', test_cmd=None, env_schema=None,
            hooks=None, lfs=None, workflows=None, workflow_preset=None,
            deploy_cmd=None, path='.'):
    "Create or update .gheasy/config.json."
    branch = branch or _git_branch(path)
    cfg = GheasyConfig(app=name, health_path=health_path, test_cmd=test_cmd, env_schema=env_schema or {},
        hooks=hooks or {}, lfs=lfs or [], workflows=workflows or {}, workflow_preset=workflow_preset,deploy_cmd=deploy_cmd)
    cfg.add_env(env, host, domain, branch=branch, srv_path=srv_path)
    cfg.save(path)
    print(f'Config written to {cfg_path(path)}')
    return cfg_path(path)

In [ ]:
#| export
def gh_add_env(env, host, domain, branch=None, srv_path='/srv/app', path='.'):
    "Add or update an environment in config. Regenerates workflow locally."
    cfg = GheasyConfig.load(path)
    cfg.add_env(env, host, domain, branch=branch or _git_branch(path), srv_path=srv_path)
    cfg.save(path)
    wf_path = Path(path) / '.github' / 'workflows' / 'gheasy.yml'
    wf_path.parent.mkdir(parents=True, exist_ok=True)
    wf_path.write_text(mk_workflow(cfg))
    print(f'Added env {env!r}. Run gh_workflow() to push to GitHub.')

def gh_add_job(name, job, path='.'):
    "Add a custom GitHub Actions job dict to config. Run gh_workflow() to push."
    cfg = GheasyConfig.load(path)
    cfg.add_job(name, job).save(path)
    print(f'Added job {name!r}. Run gh_workflow() to push.')

def gh_workflow(token=None, path='.'):
    "Write workflow YAML locally. With token, also uploads via ghapi."
    cfg = GheasyConfig.load(path)
    wf_str = mk_workflow(cfg)
    wf_path = Path(path) / '.github' / 'workflows' / 'gheasy.yml'
    wf_path.parent.mkdir(parents=True, exist_ok=True)
    wf_path.write_text(wf_str)
    print(f'Workflow written to {wf_path}')
    if token:
        import base64
        owner, repo, api = _gh_api(token, path)
        content = base64.b64encode(wf_str.encode()).decode()
        try:
            existing = api.repos.get_content(path='.github/workflows/gheasy.yml')
            api.repos.create_or_update_file_contents(
                path='.github/workflows/gheasy.yml',
                message='chore: update gheasy workflow',
                content=content, sha=existing.sha)
        except Exception:
            api.repos.create_or_update_file_contents(
                path='.github/workflows/gheasy.yml',
                message='chore: add gheasy workflow',
                content=content)
        print('Workflow uploaded to GitHub.')

def gh_status(path='.'):
    "Print deploy state table."
    if not cfg_path(path).exists():
        print('No .gheasy/config.json found.')
        return
    cfg = GheasyConfig.load(path)
    if not cfg.deploys:
        print('No deploys recorded yet.')
        return
    print(f'{"Env":<12} {"Domain":<30} {"Version":<10} {"Deployed At":<25} {"Status"}')
    print('-' * 90)
    for env_name, d in cfg.deploys.items():
        env_cfg = cfg.envs.get(env_name, EnvConfig(host='?', domain='?'))
        print(f'{env_name:<12} {env_cfg.domain:<30} {d.get("version","?")[:8]:<10} '
              f'{d.get("deployed_at","?"):<25} {d.get("status","?")}')

def gh_ship(token=None, path='.'):
    "Push workflow YAML (optionally to GitHub API) then print status."
    gh_workflow(token=token, path=path)
    gh_status(path=path)

def gh_record_deploy(env, sha, path='.'):
    "Record deploy state to config. Called by CI after successful deploy."
    cfg = GheasyConfig.load(path)
    cfg.deploys[env] = {
        'version': sha,
        'deployed_at': datetime.now(timezone.utc).isoformat(),
        'status': 'active',
    }
    cfg.save(path)

In [ ]:
with GheasyTestRepo() as t:
    gh_init('myapp', '1.2.3.4', 'myapp.com', path=t.path)
    assert cfg_path(t.path).exists()
    cfg = GheasyConfig.load(t.path)
    assert cfg.app == 'myapp'
    assert 'prod' in cfg.envs

with GheasyTestRepo() as t:
    gh_init('myapp', '1.2.3.4', 'myapp.com', path=t.path)
    gh_add_env('staging', '5.6.7.8', 'staging.myapp.com', path=t.path)
    cfg = GheasyConfig.load(t.path)
    assert 'staging' in cfg.envs
    wf_path = t.path / '.github' / 'workflows' / 'gheasy.yml'
    assert wf_path.exists()
    # no deploy_cmd set → workflow exists but no deploy job
    assert '# Managed by gheasy' in wf_path.read_text()

with GheasyTestRepo() as t:
    # with deploy_cmd → deploy job included in workflow
    gh_init('myapp', '1.2.3.4', 'myapp.com', path=t.path, deploy_cmd='./deploy.sh')
    gh_add_env('staging', '5.6.7.8', 'staging.myapp.com', path=t.path)
    wf_path = t.path / '.github' / 'workflows' / 'gheasy.yml'
    assert 'deploy-staging' in wf_path.read_text()

with GheasyTestRepo() as t:
    gh_init('myapp', '1.2.3.4', 'myapp.com', path=t.path)
    gh_workflow(path=t.path)
    wf_path = t.path / '.github' / 'workflows' / 'gheasy.yml'
    assert '# Managed by gheasy' in wf_path.read_text()

print('lifecycle OK')

Config written to /var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpy2_snpbc/.gheasy/config.json
Config written to /var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpqcjb8sni/.gheasy/config.json
Added env 'staging'. Run gh_workflow() to push to GitHub.
Config written to /var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpp_uw6mrs/.gheasy/config.json
Added env 'staging'. Run gh_workflow() to push to GitHub.
Config written to /var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmp2r8cv67b/.gheasy/config.json
Workflow written to /var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmp2r8cv67b/.github/workflows/gheasy.yml
lifecycle OK


## Composite workflows

`gh_setup()` is the one-shot setup command: init config, create GitHub environments, push workflow.
`gh_ship()` is the daily-use command: push workflow + print status.

**Typical first-time setup:**
```python
gh_setup('myapp', '1.2.3.4', 'myapp.example.com',
         env_schema={'DB_URL': None, 'PORT': '8000'},
         workflows={'lint': True})
# → creates .fastops/config.json
# → writes .github/workflows/gheasy.yml
# → installs pre-commit hook: uv run nbdev_prepare
# → prints: gh secret set VPS_SSH_KEY ...
```

**After any config change:**
```python
gh_add_env('staging', '5.6.7.8', 'staging.myapp.com', branch='dev')
gh_ship()  # push updated workflow + print status
```

In [ ]:
#| export
def gh_setup(name, host, domain, env='prod', branch=None, srv_path='/srv/app',
             test_cmd=None, env_schema=None, token=None,
             hooks=None, hook_cmd='uv run nbdev_prepare',
             lfs=None, workflows=None, workflow_preset=None,
             deploy_cmd=None, path='.'):
    """One-call setup: gh_init + gh_workflow + gh_hooks.
    Set hooks={} to skip hook installation."""
    gh_init(name, host, domain, env=env, branch=branch, srv_path=srv_path,
            health_path='/health', test_cmd=test_cmd, env_schema=env_schema,
            hooks=hooks or {}, lfs=lfs, workflows=workflows,
            workflow_preset=workflow_preset, deploy_cmd=deploy_cmd, path=path)
    gh_workflow(token=token, path=path)
    if hooks is None: gh_hooks({'pre-commit': hook_cmd}, path=path)
    elif hooks: gh_hooks(hooks, path=path)
    return cfg_path(path)

In [ ]:
with GheasyTestRepo() as t:
    result = gh_setup('myapp', '1.2.3.4', 'myapp.com', path=t.path)
    assert result == cfg_path(t.path)
    assert cfg_path(t.path).exists()
    assert (t.path / '.github' / 'workflows' / 'gheasy.yml').exists()
    hook = (t.path / '.git' / 'hooks' / 'pre-commit').read_text()
    assert 'nbdev_prepare' in hook

with GheasyTestRepo() as t:
    gh_setup('myapp', '1.2.3.4', 'myapp.com', hooks={}, path=t.path)
    assert not (t.path / '.git' / 'hooks' / 'pre-commit').exists()

print('gh_setup OK')

Config written to /var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpwejch4qh/.gheasy/config.json
Workflow written to /var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpwejch4qh/.github/workflows/gheasy.yml
Config written to /var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpvfpvdh8u/.gheasy/config.json
Workflow written to /var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpvfpvdh8u/.github/workflows/gheasy.yml
gh_setup OK


## Repo Health

## RepoFinding — structured health check result

Each finding from `GheasyRepo.check()` is a `RepoFinding` with a level, category, message,
optional auto-fix callable, human-readable command repr, and docs link.

Levels: `ok`, `warn`, `error`

In [ ]:
#| export
FINDING_BRANCH_PROTECTION = 'branch-protection'
FINDING_DEPENDABOT        = 'dependabot'
FINDING_TOPICS            = 'topics'
FINDING_LFS               = 'lfs'
FINDING_HOOKS             = 'hooks'
FINDING_BUILD             = 'build'
FINDING_WORKFLOWS         = 'workflows'

@dataclass
class RepoFinding:
    "A single finding from a repo health check."
    level: str      # 'ok' | 'warn' | 'error'
    category: str
    message: str
    fix: Any = None
    cmd_repr: str = ''
    docs: str = ''

    def __repr__(self):
        icon = {'ok': '\u2713', 'warn': '\u26a0', 'error': '\u2717'}.get(self.level, '?')
        hint = f' \u2192 {self.cmd_repr}' if self.cmd_repr else ''
        return f'{icon} [{self.category}] {self.message}{hint}'

    def apply(self):
        if self.fix: self.fix()
        else: print(f'No auto-fix for {self.category}: {self.cmd_repr}')

In [ ]:
#| export
def gh_check(owner=None, repo=None, token=None, path='.', remote=True, local=True, verbose=True):
    "Audit repo health. local: hooks/gitattributes/pyproject. remote: protection/topics/dependabot."
    chks = []
    if local:
        hp = Path(path) / '.git' / 'hooks' / 'pre-commit'
        if hp.exists() and 'gheasy-managed' in hp.read_text():
            chks.append(RepoFinding('ok', FINDING_HOOKS, 'pre-commit hook installed'))
        else: chks.append(RepoFinding('warn', FINDING_HOOKS, 'No gheasy pre-commit hook',
              fix=lambda: gh_githooks_pre_commit(path=path), cmd_repr='gh_githooks_pre_commit()'))

        ga = Path(path) / '.gitattributes'
        if ga.exists() and '# BEGIN gheasy-lfs' in ga.read_text():
            chks.append(RepoFinding('ok', FINDING_LFS, '.gitattributes has LFS block'))
        else: chks.append(RepoFinding('warn', FINDING_LFS, 'No gheasy LFS block in .gitattributes',
            fix=lambda: gh_gitattributes(path=path), cmd_repr='gh_gitattributes()'))

        pp = Path(path) / 'pyproject.toml'
        if pp.exists():
            if 'hatchling' in pp.read_text():
	            chks.append(RepoFinding('ok', FINDING_BUILD, 'pyproject.toml uses hatchling'))
            else: chks.append(RepoFinding('warn', FINDING_BUILD, 'pyproject.toml not using hatchling',
                fix=lambda: gh_pyproject_to_hatchling(path=path), cmd_repr='gh_pyproject_to_hatchling()'))

    if remote and (token or _resolve_gh_token()):
        try:
            if owner is None or repo is None: owner, repo = _get_repo_slug(path)
            api = GhApi(owner=owner, repo=repo, token=token or _resolve_gh_token())
            repo_data = api.repos.get()
            if repo_data.topics: chks.append(RepoFinding('ok', FINDING_TOPICS, f'Topics set: {repo_data.topics}'))
            else: chks.append(RepoFinding('warn', FINDING_TOPICS, 'No topics set', cmd_repr='gh_topics([...])'))
            try:
                api.repos.get_content(path='.github/dependabot.yml')
                chks.append(RepoFinding('ok', FINDING_DEPENDABOT, 'dependabot.yml exists'))
            except: chks.append(RepoFinding('warn', FINDING_DEPENDABOT, 'No dependabot.yml',
                                                cmd_repr='write .github/dependabot.yml'))
        except Exception as e: chks.append(RepoFinding('warn', FINDING_BRANCH_PROTECTION, f'Could not check remote: {e}'))

    if verbose: L(chks).map(print)
    return chks

def gh_apply(findings, dry_run=False, confirm=False):
    "Auto-fix all non-ok findings that have a fix callable."
    applied = []
    for f in findings:
        if f.level == 'ok' or f.fix is None: continue
        if dry_run: print(f'[dry-run] would fix: {f.category} \u2192 {f.cmd_repr}'); continue
        if confirm:
            resp = input(f'Fix {f.category}? ({f.cmd_repr}) [y/N] ')
            if resp.lower() != 'y': continue
        f.apply()
        applied.append(f)
    return applied

## GheasyRepo — Repo lifecycle management

`GheasyRepo` is a lazy handle to a GitHub repo. It makes **zero API calls on instantiation** —
only when you call `check()` or `apply_all()`.

- Pass any of: `owner/repo`, a GitHub URL, or a local path with a remote configured
- `check()` audits the repo: branch protection, dependabot, topics, secrets, LFS, hooks
- `apply_all()` auto-fixes everything it can; dry_run=True to preview
- `create()` creates the remote repo via GitHub API
- `new(template='nbdev')` scaffolds a full project: create → clone → nbdev-new → migrate pyproject → push

`GHEASY_DEFAULTS` encodes 2025 best practices as zero-config defaults.

In [ ]:
#| export
@dataclass
class GheasyRepo:
    "Thin stateful shell over gh_* functions. All logic lives in standalone functions."
    ref: str = None
    token: str = None
    path: str = '.'

    def __post_init__(self):
        if self.ref: self._owner, self._repo = _resolve_gh_repo_input(self.ref, self.path)
        else: self._owner, self._repo = None, None

    @property
    def full_name(self): return f'{self._owner}/{self._repo}' if self._owner and self._repo else None

    def check(self, remote=True, local=True, verbose=True):
        return gh_check(self._owner, self._repo, self.token, self.path, remote, local, verbose)

    def apply(self, findings=None, dry_run=False, confirm=False):
        if findings is None: findings = self.check(verbose=False)
        return gh_apply(findings, dry_run=dry_run, confirm=confirm)

    def create(self, private=True, description='', auto_init=False):
        api = GhApi(token=self.token or _resolve_gh_token())
        return api.repos.create_for_authenticated_user(
            name=self._repo, private=private,
            description=description, auto_init=auto_init)

    @classmethod
    def new(cls, ref, template='nbdev', private=True, description='',
            token=None, parent_dir='.', topics=None, workflows=None):
        "Full project scaffold: create repo, clone, configure, push."
        owner, repo = _resolve_gh_repo_input(ref)
        inst = cls(ref=ref, token=token or _resolve_gh_token())
        inst.create(private=private, description=description)
        sp.run(['git', 'clone', f'https://github.com/{owner}/{repo}', str(Path(parent_dir) / repo)], check=True)
        local_path = Path(parent_dir) / repo
        if template == 'nbdev': sp.run(['uv', 'run', 'nbdev-new'], cwd=local_path, check=True)
        gh_pyproject_to_hatchling(path=local_path)
        gh_lfs(persist=True, path=local_path)
        gh_githooks_pre_commit(path=local_path)
        if workflows: gh_init(repo, '', '', workflows=workflows, path=local_path)
        gh_workflow(token=inst.token, path=local_path)
        sp.run(['git', 'add', '-A'], cwd=local_path, check=True)
        sp.run(['git', 'commit', '-m', 'chore: gheasy init'], cwd=local_path, check=True)
        sp.run(['git', 'push', '-u', 'origin', 'HEAD'], cwd=local_path, check=True)
        if topics: gh_topics(topics, token=inst.token, path=local_path)
        return cls(ref=ref, token=inst.token, path=str(local_path))

In [ ]:
with GheasyTestRepo() as t:
    findings = gh_check(path=t.path, remote=False)
    levels = [f.level for f in findings]
    assert 'warn' in levels

    gh_githooks_pre_commit(path=t.path)
    findings2 = gh_check(path=t.path, remote=False, verbose=False)
    hook_findings = [f for f in findings2 if f.category == FINDING_HOOKS]
    assert hook_findings[0].level == 'ok'

with GheasyTestRepo() as t:
    GheasyConfig(app='test').set_lfs(['*.pt']).save(t.path)
    findings = gh_check(path=t.path, remote=False, verbose=False)
    warn_before = sum(1 for f in findings if f.level == 'warn')
    gh_apply(findings)
    findings2 = gh_check(path=t.path, remote=False, verbose=False)
    warn_after = sum(1 for f in findings2 if f.level == 'warn')
    assert warn_after < warn_before

print('repo health OK')

⚠ [hooks] No gheasy pre-commit hook → gh_githooks_pre_commit()
⚠ [lfs] No gheasy LFS block in .gitattributes → gh_gitattributes()
repo health OK


## Migration

## pyproject.toml migration

`migrate_pyproject_to_hatchling()` upgrades a `pyproject.toml` from setuptools/flit to hatchling,
adds `uv` as the package manager hint, and removes legacy build artifacts.
Uses `tomlkit` for round-trip TOML editing (preserves comments and formatting).


In [ ]:
#| export
def _migrate_pyproject_to_hatchling(path='.'):
    "Convert pyproject.toml to hatchling. Returns True if changes made."
    import tomlkit
    pp = Path(path) / 'pyproject.toml'
    if not pp.exists(): return False
    doc = tomlkit.parse(pp.read_text())
    changed = False
    bs = doc.setdefault('build-system', {})
    if bs.get('build-backend') != 'hatchling.build':
        bs['build-backend'] = 'hatchling.build'
        bs['requires'] = ['hatchling']
        changed = True
    name = doc.get('project', {}).get('name', '')
    if name:
        targets = doc.setdefault('tool', {}).setdefault('hatch', {}).setdefault('build', {}).setdefault('targets', {})
        wheel = targets.setdefault('wheel', {})
        if wheel.get('packages') != [name]:
            wheel['packages'] = [name]
            changed = True
        sdist = targets.setdefault('sdist', {})
        sdist_includes = [f'/{name}', '/README.md', '/pyproject.toml']
        if sdist.get('include') != sdist_includes:
            sdist['include'] = sdist_includes
            changed = True
    for key in ('setuptools', 'setup-tools'):
        if key in doc.get('tool', {}):
            del doc['tool'][key]
            changed = True
    if changed: pp.write_text(tomlkit.dumps(doc))
    return changed

def gh_pyproject_to_hatchling(path='.', dry_run=False):
    "Convert pyproject.toml to hatchling build backend. Idempotent."
    if dry_run:
        print(f'[dry-run] Would migrate {Path(path) / "pyproject.toml"} to hatchling')
        return
    changed = _migrate_pyproject_to_hatchling(path)
    print('Migrated to hatchling.' if changed else 'Already using hatchling (no changes).')

In [ ]:
with GheasyTestRepo() as t:
    (t.path / 'pyproject.toml').write_text('''
[build-system]
requires = ["setuptools"]
build-backend = "setuptools.build_meta"

[project]
name = "myapp"
''')
    changed = _migrate_pyproject_to_hatchling(t.path)
    assert changed
    import tomlkit
    doc = tomlkit.parse((t.path / 'pyproject.toml').read_text())
    assert doc['build-system']['build-backend'] == 'hatchling.build'
    targets = doc['tool']['hatch']['build']['targets']
    assert targets['wheel']['packages'] == ['myapp']
    assert targets['sdist']['include'] == ['/myapp', '/README.md', '/pyproject.toml']
    changed2 = _migrate_pyproject_to_hatchling(t.path)
    assert not changed2, 'should be idempotent'

print('migration OK')

migration OK


## CLI

In [ ]:
#| export
from cyclopts import App as _App

app = _App(name='gheasy', help='GitHub made easy \u2014 git workflows, CI/CD, hooks, LFS.')

app.command(gh_setup)
app.command(gh_init)
app.command(gh_add_env)
app.command(gh_add_job)
app.command(gh_workflow)
app.command(gh_ship)
app.command(gh_status)
app.command(gh_check)
app.command(gh_apply)
app.command(gh_hooks)
app.command(gh_lfs)
app.command(gh_gitattributes)
app.command(gh_protect)
app.command(gh_topics)
app.command(gh_push_env)
app.command(gh_secrets_from_file)
app.command(gh_pyproject_to_hatchling)

def main(): app()

In [ ]:
import inspect
assert inspect.isfunction(main)
assert callable(app)
print('CLI OK')

CLI OK


In [ ]:
#| eval: False
# These require gh CLI + GitHub authentication — run manually
# gh_secret('MY_API_KEY', 'abc123')
# gh_secret('PROD_DB_URL', 'postgres://...', env='prod')  # scoped to prod environment
# gh_secrets_from_file('.env.production', env='prod', dry_run=True)
print('gh_secret / gh_secrets_from_file — run manually with gh CLI')

gh_secret / gh_secrets_from_file — run manually with gh CLI


In [ ]:
#| eval: False
# These require gh CLI + GitHub authentication + an open repo — run manually
# url = gh_pr('feat: my feature', body='Closes #42')
# print(url)
# gh_pr_merge(squash=True)
print('gh_pr / gh_pr_merge — run manually with gh CLI')

gh_pr / gh_pr_merge — run manually with gh CLI


In [ ]:
#| eval: False
# These require a GitHub token with admin scope — run manually
# gh_protect('main', require_reviews=1, require_status_checks=['test'])
# gh_topics(['python', 'devops', 'ci-cd'])
print('gh_protect / gh_topics — require GitHub token with admin scope')

In [ ]:
#| eval: False
# Full remote lifecycle — requires GITHUB_TOKEN with repo scope
# Run manually: GheasyRepo.new('your-org/gheasy-test-repo')
import os
if os.environ.get('GITHUB_TOKEN'):
    with GheasyTestRepo() as r:
        # Remote check on a known public repo
        repo = GheasyRepo('AnswerDotAI/fastcore', token=os.environ['GITHUB_TOKEN'])
        findings = repo.check(local=False)
        cats = {f.category for f in findings}
        assert FINDING_BRANCH_PROTECTION in cats
        print(f'Remote check: {len(findings)} findings on AnswerDotAI/fastcore')
else:
    print('Skipped: GITHUB_TOKEN not set')


✓ [topics] Topics set: ['data-structures', 'developer-tools', 'dispatch', 'documentation-generator', 'fastai', 'functional-programming', 'languages', 'parallel-processing', 'python']
⚠ [dependabot] No dependabot.yml → write .github/dependabot.yml


AssertionError: 

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()